In [27]:
import cv2
import numpy as np
import mediapipe as mp
import math
import random
import time
from collections import deque

mp_hands = mp.solutions.hands

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

### Vector tulip shape: multi-color, 3D spin, golden borders

In [34]:
# Different color palettes (dark_shade, light_shade) in BGR — vibrant reds, pinks, magenta
TULIP_PALETTES = [
    ((40,  20, 170), (70,  45, 225)),   # deep red / bright red
    ((140, 25, 200), (180, 70, 255)),   # hot pink
    ((150, 30, 180), (195, 70, 230)),   # magenta-purple
    ((25,  55, 215), (45,  90, 250)),   # coral red-orange
    ((110, 55, 215), (150, 95, 255)),   # rose pink
]

GOLD = (25, 215, 255)  # BGR gold for petal borders


def petal_points(height, width):
    return np.array([
        [0,           0],
        [ width*0.50, -height*0.15],
        [ width*0.45, -height*0.55],
        [ width*0.15, -height],
        [-width*0.15, -height],
        [-width*0.45, -height*0.55],
        [-width*0.50, -height*0.15],
    ], dtype=np.float32)


def rotate_points(points, angle_deg):
    theta = math.radians(angle_deg)
    c, s = math.cos(theta), math.sin(theta)
    rot = np.array([[c, -s], [s, c]], dtype=np.float32)
    return points @ rot.T


def draw_tulip_glow(glow_layer, tip_pos, angle_deg, bloom, size, palette, self_rotation_deg):
    """Draws a glowing, multi-tone tulip with golden borders. self_rotation_deg
    drives a squash-based pseudo-3D spin illusion (like a coin turning)."""
    if bloom <= 0.03:
        return

    h = size * (0.35 + 0.65 * bloom)
    w = size * 0.55 * (0.35 + 0.65 * bloom)

    # Pseudo-3D spin: squashing width by cos(angle) fakes a turning flower
    squash = max(0.15, abs(math.cos(math.radians(self_rotation_deg))))
    shade = 0.55 + 0.45 * squash  # darker when "edge-on", brighter when facing camera

    dark_color, light_color = palette
    dark_shaded = tuple(c * shade for c in dark_color)
    light_shaded = tuple(c * shade for c in light_color)

    lobes = [(-20, dark_shaded), (0, light_shaded), (20, dark_shaded)]

    for tilt, color in lobes:
        pts = petal_points(h, w)
        pts[:, 0] *= squash              # apply 3D spin squash
        pts = rotate_points(pts, tilt)
        pts = rotate_points(pts, angle_deg)
        pts[:, 0] += tip_pos[0]
        pts[:, 1] += tip_pos[1]
        pts_int = pts.astype(np.int32)

        cv2.fillPoly(glow_layer, [pts_int], color, lineType=cv2.LINE_AA)
        cv2.polylines(glow_layer, [pts_int], True, GOLD, 2, cv2.LINE_AA)  # golden border

    base_glow_r = max(2, int(size * 0.10 * bloom))
    cv2.circle(glow_layer, (int(tip_pos[0]), int(tip_pos[1])), base_glow_r, GOLD, -1, cv2.LINE_AA)

### Pinch-distance metric (thumb ↔ index only)

In [35]:
def hand_pinch_metric(hand_landmarks, image_w, image_h):
    """Returns (pinch_progress 0-1, wrist_pos, thumb_tip, index_tip) for one hand.
    Progress is based purely on the distance between thumb tip and index tip,
    normalized by hand size so it works at any distance from the camera."""
    pts = np.array([(lm.x * image_w, lm.y * image_h) for lm in hand_landmarks.landmark])

    wrist = pts[0]
    middle_mcp = pts[9]
    hand_scale = np.linalg.norm(wrist - middle_mcp) + 1e-6

    thumb_tip = pts[4]
    index_tip = pts[8]
    pinch_dist = np.linalg.norm(thumb_tip - index_tip) / hand_scale

    progress = (pinch_dist - 0.2) / (1.6 - 0.2)
    progress = float(np.clip(progress, 0, 1))

    return progress, wrist, thumb_tip, index_tip

### Magic sparkle trail (glowing dust)

In [36]:
def draw_star(layer, center, size, color):
    cx, cy = int(center[0]), int(center[1])
    length = max(1, int(size * 2))
    thickness = max(1, int(size * 0.35))
    cv2.line(layer, (cx - length, cy), (cx + length, cy), color, thickness, cv2.LINE_AA)
    cv2.line(layer, (cx, cy - length), (cx, cy + length), color, thickness, cv2.LINE_AA)


class MagicSparkles:
    def __init__(self):
        self.particles = []

    def spawn(self, pos, n=1):
        for _ in range(n):
            self.particles.append({
                'pos': [pos[0] + random.uniform(-10, 10), pos[1] + random.uniform(-10, 10)],
                'vel': [random.uniform(-0.4, 0.4), random.uniform(-1.2, -0.3)],
                'size': random.uniform(2, 6),
                'life': 1.0,
                'decay': random.uniform(0.02, 0.04),
                'color': random.choice([(170, 210, 255), (230, 230, 255), (200, 170, 255)])
            })

    def update(self):
        self.particles = [p for p in self.particles if p['life'] > 0]
        for p in self.particles:
            p['pos'][0] += p['vel'][0]
            p['pos'][1] += p['vel'][1]
            p['life'] -= p['decay']

    def draw(self, glow_layer):
        for p in self.particles:
            alpha = np.clip(p['life'], 0, 1)
            color = tuple(c * alpha for c in p['color'])
            draw_star(glow_layer, p['pos'], p['size'], color)

### Bouquet renderer (Grow = height, Bloom = petal opening, each stem spins + has its own color)

In [37]:
def get_stem_configs(n=5):
    configs = []
    for i in range(n):
        t = (i / (n - 1)) - 0.5   # -0.5 .. 0.5
        configs.append({
            'x_offset': t * 90,
            'max_height': 300 - abs(t) * 60,
            'angle': t * 30,
            'palette': TULIP_PALETTES[i % len(TULIP_PALETTES)],
            'spin_speed': random.uniform(40, 80),      # degrees/sec, varies per stem
            'spin_phase': random.uniform(0, 360),      # so stems don't spin in sync
        })
    return configs

STEM_CONFIGS = get_stem_configs(5)

def draw_bouquet(glow_layer, anchor, grow, bloom, elapsed_time):
    for cfg in STEM_CONFIGS:
        base_x = anchor[0] + cfg['x_offset'] * 0.4
        base_y = anchor[1]
        stem_len = cfg['max_height'] * grow
        angle_rad = math.radians(cfg['angle'])

        tip_x = base_x + math.sin(angle_rad) * stem_len
        tip_y = base_y - math.cos(angle_rad) * stem_len

        cv2.line(glow_layer, (int(base_x), int(base_y)), (int(tip_x), int(tip_y)),
                  (200, 160, 60), 2, cv2.LINE_AA)

        self_rotation_deg = (elapsed_time * cfg['spin_speed'] + cfg['spin_phase']) % 360

        draw_tulip_glow(
            glow_layer, (tip_x, tip_y), cfg['angle'], bloom,
            size=110, palette=cfg['palette'], self_rotation_deg=self_rotation_deg
        )

### Main loop using pinch distance + visual line feedback

In [38]:
def run_tulip_bouquet(camera_index=0, smoothing=0.15, glow_intensity=1.0):
    cap = cv2.VideoCapture(camera_index)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    grow_smoothed = 0.0
    bloom_smoothed = 0.0
    sparkles = MagicSparkles()
    prev_time = time.time()
    start_time = time.time()

    print("Left side of frame = Bloom hand | Right side of frame = Grow hand.")
    print("Pinch thumb + index apart to increase each value. Press 'q' to quit.")

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.flip(frame, 1)
        h, w, _ = frame.shape

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb)

        anchor = (int(w * 0.25), int(h * 0.85))
        glow = np.zeros_like(frame, dtype=np.float32)

        grow_raw, bloom_raw = 0.0, 0.0
        grow_info, bloom_info = None, None

        if results.multi_hand_landmarks:
            hand_infos = []
            for hand_landmarks in results.multi_hand_landmarks:
                progress, wrist, thumb_tip, index_tip = hand_pinch_metric(hand_landmarks, w, h)
                hand_infos.append((wrist[0], progress, thumb_tip, index_tip))

            hand_infos.sort(key=lambda item: item[0])  # leftmost = Bloom, rightmost = Grow

            if len(hand_infos) == 1:
                _, progress, thumb_tip, index_tip = hand_infos[0]
                grow_raw = bloom_raw = progress
                grow_info = bloom_info = (progress, thumb_tip, index_tip)
            else:
                _, bloom_raw, b_thumb, b_index = hand_infos[0]
                bloom_info = (bloom_raw, b_thumb, b_index)
                _, grow_raw, g_thumb, g_index = hand_infos[-1]
                grow_info = (grow_raw, g_thumb, g_index)

        grow_smoothed += (grow_raw - grow_smoothed) * smoothing
        bloom_smoothed += (bloom_raw - bloom_smoothed) * smoothing

        elapsed = time.time() - start_time
        draw_bouquet(glow, anchor, grow_smoothed, bloom_smoothed, elapsed)

        if bloom_smoothed > 0.85 and random.random() < 0.4:
            sparkles.spawn(anchor, n=2)

        sparkles.update()
        sparkles.draw(glow)

        glow_soft = cv2.GaussianBlur(glow, (0, 0), sigmaX=10)
        glow_sharp = cv2.GaussianBlur(glow, (0, 0), sigmaX=2)
        combined_glow = (glow_sharp * 0.6 + glow_soft * 0.6) * glow_intensity

        output = np.clip(frame.astype(np.float32) + combined_glow, 0, 255).astype(np.uint8)

        if grow_info is not None:
            progress, thumb_tip, index_tip = grow_info
            p1 = (int(thumb_tip[0]), int(thumb_tip[1]))
            p2 = (int(index_tip[0]), int(index_tip[1]))
            cv2.line(output, p1, p2, (255, 180, 80), 2, cv2.LINE_AA)
            cv2.circle(output, p1, 5, (255, 180, 80), -1, cv2.LINE_AA)
            cv2.circle(output, p2, 5, (255, 180, 80), -1, cv2.LINE_AA)
            mid = ((p1[0] + p2[0]) // 2, (p1[1] + p2[1]) // 2)
            cv2.putText(output, f"Grow: {grow_smoothed:.2f}", (mid[0] + 10, mid[1]),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 220, 180), 2, cv2.LINE_AA)

        if bloom_info is not None:
            progress, thumb_tip, index_tip = bloom_info
            p1 = (int(thumb_tip[0]), int(thumb_tip[1]))
            p2 = (int(index_tip[0]), int(index_tip[1]))
            cv2.line(output, p1, p2, (200, 120, 255), 2, cv2.LINE_AA)
            cv2.circle(output, p1, 5, (200, 120, 255), -1, cv2.LINE_AA)
            cv2.circle(output, p2, 5, (200, 120, 255), -1, cv2.LINE_AA)
            mid = ((p1[0] + p2[0]) // 2, (p1[1] + p2[1]) // 2)
            cv2.putText(output, f"Bloom: {bloom_smoothed:.2f}", (mid[0] + 10, mid[1]),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 180, 220), 2, cv2.LINE_AA)

        curr_time = time.time()
        fps = 1 / (curr_time - prev_time + 1e-6)
        prev_time = curr_time
        cv2.putText(output, f"FPS: {int(fps)}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        cv2.imshow("Tulip Bouquet - Grow & Bloom", output)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

### Run it

In [39]:
run_tulip_bouquet(camera_index=0, smoothing=0.15, glow_intensity=1.0)

Left side of frame = Bloom hand | Right side of frame = Grow hand.
Pinch thumb + index apart to increase each value. Press 'q' to quit.
